# Source of Hire Effectiveness Analysis

## Objective
Determine which recruiting channels produce the best hires by analyzing:
- Quality of hire by source (performance, retention, manager satisfaction)
- Cost and time efficiency by source
- Volume and conversion rates
- ROI calculation for sourcing budget allocation

## Key Questions
1. Which sources produce the highest quality hires?
2. Which sources are most cost-effective?
3. Which sources have the best conversion rates?
4. How should we reallocate our recruiting budget?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries loaded successfully")

## 1. Load Data

In [ ]:
# Load datasets
hires_df = pd.read_csv('../data/hires.csv')
candidates_df = pd.read_csv('../data/candidates.csv')
requisitions_df = pd.read_csv('../data/requisitions.csv')
interviews_df = pd.read_csv('../data/interviews.csv')

print(f"📊 Data loaded:")
print(f"   Hires: {len(hires_df):,} records")
print(f"   Candidates: {len(candidates_df):,} records")
print(f"   Requisitions: {len(requisitions_df):,} records")
print(f"   Interviews: {len(interviews_df):,} records")

# Preview hires data
print("\n📋 Hires data preview:")
hires_df.head()

## 2. Source Volume & Conversion Analysis

In [ ]:
# Calculate application volume and hire rate by source
source_metrics = candidates_df.groupby('source').agg({
    'candidate_id': 'count'
}).rename(columns={'candidate_id': 'applications'})

# Add hire counts
hired_by_source = candidates_df[candidates_df['current_stage'] == 'Hired'].groupby('source').size()
source_metrics['hires'] = hired_by_source
source_metrics['hires'] = source_metrics['hires'].fillna(0).astype(int)

# Calculate conversion rate
source_metrics['hire_rate_%'] = (source_metrics['hires'] / source_metrics['applications'] * 100).round(2)

# Sort by volume
source_metrics = source_metrics.sort_values('applications', ascending=False)

print("📈 Source Volume & Conversion Rates:")
print(source_metrics)
print(f"\n📊 Overall hire rate: {(source_metrics['hires'].sum() / source_metrics['applications'].sum() * 100):.2f}%")

In [ ]:
# Visualize conversion funnel by source
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Applications by source
source_metrics['applications'].plot(kind='barh', ax=ax1, color='steelblue')
ax1.set_xlabel('Number of Applications')
ax1.set_title('Application Volume by Source')
ax1.grid(axis='x', alpha=0.3)

# Hire rate by source
source_metrics['hire_rate_%'].plot(kind='barh', ax=ax2, color='green')
ax2.set_xlabel('Hire Rate (%)')
ax2.set_title('Conversion Rate (Application → Hire) by Source')
ax2.axvline(x=(source_metrics['hires'].sum() / source_metrics['applications'].sum() * 100), 
            color='red', linestyle='--', label='Overall Avg', alpha=0.7)
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Quality of Hire by Source

We'll analyze multiple quality indicators:
- Performance ratings at 12 months
- 12-month retention rate
- Manager satisfaction scores
- Time to productivity

In [ ]:
# Quality metrics by source
quality_by_source = hires_df.groupby('source').agg({
    'performance_rating_12mo': ['mean', 'std', 'count'],
    'still_employed_12mo': 'mean',
    'manager_satisfaction': 'mean',
    'days_to_productivity': 'mean'
}).round(2)

# Flatten column names
quality_by_source.columns = ['_'.join(col).strip() for col in quality_by_source.columns.values]
quality_by_source = quality_by_source.rename(columns={
    'performance_rating_12mo_mean': 'avg_performance',
    'performance_rating_12mo_std': 'performance_std',
    'performance_rating_12mo_count': 'n_hires',
    'still_employed_12mo_mean': 'retention_rate',
    'manager_satisfaction_mean': 'avg_satisfaction',
    'days_to_productivity_mean': 'avg_days_to_prod'
})

quality_by_source['retention_rate'] = (quality_by_source['retention_rate'] * 100).round(1)
quality_by_source = quality_by_source.sort_values('avg_performance', ascending=False)

print("⭐ Quality of Hire by Source:")
print(quality_by_source[['n_hires', 'avg_performance', 'retention_rate', 'avg_satisfaction']])

In [ ]:
# Visualize quality metrics by source
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Filter sources with at least 3 hires for statistical reliability
reliable_sources = quality_by_source[quality_by_source['n_hires'] >= 3]

# Performance rating
ax = axes[0, 0]
bars = ax.barh(reliable_sources.index, reliable_sources['avg_performance'], color='steelblue')
ax.axvline(x=hires_df['performance_rating_12mo'].mean(), color='red', linestyle='--', alpha=0.7, label='Overall Avg')
ax.set_xlabel('Average Performance Rating (12 months)')
ax.set_title('Performance Rating by Source')
ax.set_xlim(1, 5)
ax.legend()
ax.grid(axis='x', alpha=0.3)

# Add sample size labels
for i, (idx, row) in enumerate(reliable_sources.iterrows()):
    ax.text(row['avg_performance'], i, f"  n={int(row['n_hires'])}", 
            va='center', fontsize=9, color='black')

# Retention rate
ax = axes[0, 1]
bars = ax.barh(reliable_sources.index, reliable_sources['retention_rate'], color='green')
ax.axvline(x=(hires_df['still_employed_12mo'].mean() * 100), color='red', linestyle='--', alpha=0.7, label='Overall Avg')
ax.set_xlabel('12-Month Retention Rate (%)')
ax.set_title('Retention Rate by Source')
ax.set_xlim(0, 100)
ax.legend()
ax.grid(axis='x', alpha=0.3)

# Manager satisfaction
ax = axes[1, 0]
bars = ax.barh(reliable_sources.index, reliable_sources['avg_satisfaction'], color='orange')
ax.axvline(x=hires_df['manager_satisfaction'].mean(), color='red', linestyle='--', alpha=0.7, label='Overall Avg')
ax.set_xlabel('Manager Satisfaction Score')
ax.set_title('Manager Satisfaction by Source')
ax.set_xlim(1, 5)
ax.legend()
ax.grid(axis='x', alpha=0.3)

# Days to productivity
ax = axes[1, 1]
bars = ax.barh(reliable_sources.index, reliable_sources['avg_days_to_prod'], color='purple')
ax.axvline(x=hires_df['days_to_productivity'].mean(), color='red', linestyle='--', alpha=0.7, label='Overall Avg')
ax.set_xlabel('Days to Productivity')
ax.set_title('Time to Productivity by Source (Lower is Better)')
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n⚠️ Note: Only showing sources with 3+ hires for statistical reliability")

## 4. Statistical Significance Testing

Test whether differences in quality between sources are statistically significant.

In [ ]:
# Compare top sources to average using t-tests
top_sources = quality_by_source.head(3).index.tolist()

print("📊 Statistical Significance Tests (Performance Rating):\n")

overall_mean = hires_df['performance_rating_12mo'].mean()

for source in top_sources:
    source_data = hires_df[hires_df['source'] == source]['performance_rating_12mo']
    
    if len(source_data) >= 3:
        # One-sample t-test vs. overall mean
        t_stat, p_value = stats.ttest_1samp(source_data, overall_mean)
        
        print(f"{source}:")
        print(f"  Mean: {source_data.mean():.2f} (n={len(source_data)})")
        print(f"  vs Overall: {overall_mean:.2f}")
        print(f"  Difference: {(source_data.mean() - overall_mean):.2f}")
        print(f"  p-value: {p_value:.4f}")
        
        if p_value < 0.05:
            print(f"  ✅ Statistically significant (p < 0.05)")
        else:
            print(f"  ⚠️ Not statistically significant (p >= 0.05)")
        print()

print("\n💡 Interpretation:")
print("   p < 0.05 = Difference is statistically significant (likely not due to chance)")
print("   p >= 0.05 = Difference may be due to random variation")

## 5. Cost & Efficiency Analysis

Estimate cost-per-hire and ROI by source. Using industry benchmarks:
- Referrals: $1,000
- LinkedIn: $3,500
- Indeed: $2,500
- Career Site: $500
- Agency: $15,000
- University: $2,000
- Glassdoor: $3,000
- Other Job Board: $2,000

In [ ]:
# Cost-per-hire by source (industry benchmarks)
cost_per_hire = {
    'Referral': 1000,
    'LinkedIn': 3500,
    'Indeed': 2500,
    'Company Career Site': 500,
    'Recruiting Agency': 15000,
    'University Recruiting': 2000,
    'Glassdoor': 3000,
    'Other Job Board': 2000
}

# Merge quality and cost data
roi_analysis = quality_by_source.copy()
roi_analysis['cost_per_hire'] = roi_analysis.index.map(cost_per_hire)

# Calculate composite quality score (normalized 0-100)
# 40% performance, 30% retention, 30% manager satisfaction
roi_analysis['quality_score'] = (
    (roi_analysis['avg_performance'] / 5 * 100) * 0.4 +
    (roi_analysis['retention_rate']) * 0.3 +
    (roi_analysis['avg_satisfaction'] / 5 * 100) * 0.3
).round(1)

# Calculate ROI: Quality per dollar spent (higher is better)
roi_analysis['roi_score'] = (roi_analysis['quality_score'] / roi_analysis['cost_per_hire'] * 1000).round(2)

roi_analysis = roi_analysis.sort_values('roi_score', ascending=False)

print("💰 ROI Analysis by Source:\n")
print(roi_analysis[['n_hires', 'quality_score', 'cost_per_hire', 'roi_score']])
print("\n💡 ROI Score = (Quality Score / Cost per Hire) × 1000")
print("   Higher ROI = Better quality for the money")

In [ ]:
# Visualize Quality vs. Cost scatter plot
plt.figure(figsize=(12, 7))

# Filter for sources with sufficient data
plot_data = roi_analysis[roi_analysis['n_hires'] >= 3]

# Create scatter plot
scatter = plt.scatter(plot_data['cost_per_hire'], 
                     plot_data['quality_score'],
                     s=plot_data['n_hires'] * 50,  # Size by volume
                     alpha=0.6,
                     c=plot_data['roi_score'],
                     cmap='RdYlGn',
                     edgecolors='black',
                     linewidth=1)

# Add labels for each point
for idx, row in plot_data.iterrows():
    plt.annotate(idx, 
                xy=(row['cost_per_hire'], row['quality_score']),
                xytext=(5, 5), 
                textcoords='offset points',
                fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

# Add reference lines
plt.axhline(y=plot_data['quality_score'].median(), color='gray', linestyle='--', alpha=0.5, label='Median Quality')
plt.axvline(x=plot_data['cost_per_hire'].median(), color='gray', linestyle='--', alpha=0.5, label='Median Cost')

# Add quadrant labels
max_x = plot_data['cost_per_hire'].max()
max_y = plot_data['quality_score'].max()
median_x = plot_data['cost_per_hire'].median()
median_y = plot_data['quality_score'].median()

plt.text(median_x * 0.5, max_y * 0.95, '🏆 HIGH VALUE\n(Low Cost, High Quality)', 
         ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
plt.text(median_x * 1.5, max_y * 0.95, '⚠️ EXPENSIVE\n(High Cost, High Quality)', 
         ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))
plt.text(median_x * 0.5, max_y * 0.85, '👍 BUDGET FRIENDLY\n(Low Cost, Mid Quality)', 
         ha='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
plt.text(median_x * 1.5, max_y * 0.85, '❌ LOW VALUE\n(High Cost, Mid Quality)', 
         ha='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))

plt.xlabel('Cost per Hire ($)', fontsize=12)
plt.ylabel('Composite Quality Score (0-100)', fontsize=12)
plt.title('Quality vs. Cost by Source (bubble size = volume)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='ROI Score')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Budget Allocation Recommendations

In [ ]:
# Current vs. Recommended allocation
# Current allocation based on hire volume
current_spend = pd.DataFrame({
    'source': roi_analysis.index,
    'current_hires': roi_analysis['n_hires'],
    'cost_per_hire': roi_analysis['cost_per_hire'],
    'quality_score': roi_analysis['quality_score'],
    'roi_score': roi_analysis['roi_score']
})

current_spend['current_spend'] = current_spend['current_hires'] * current_spend['cost_per_hire']
total_spend = current_spend['current_spend'].sum()
current_spend['current_allocation_%'] = (current_spend['current_spend'] / total_spend * 100).round(1)

# Recommended allocation: Weight by ROI score
# Higher ROI = More budget (but cap at 40% per source for diversification)
current_spend['roi_weight'] = current_spend['roi_score'] / current_spend['roi_score'].sum()
current_spend['recommended_allocation_%'] = (current_spend['roi_weight'] * 100).round(1)

# Cap any source at 40% for risk diversification
current_spend.loc[current_spend['recommended_allocation_%'] > 40, 'recommended_allocation_%'] = 40
# Renormalize
current_spend['recommended_allocation_%'] = (current_spend['recommended_allocation_%'] / current_spend['recommended_allocation_%'].sum() * 100).round(1)

current_spend['recommended_spend'] = (current_spend['recommended_allocation_%'] / 100 * total_spend).round(0)
current_spend['change_$'] = (current_spend['recommended_spend'] - current_spend['current_spend']).round(0)
current_spend['change_%'] = ((current_spend['change_$'] / current_spend['current_spend']) * 100).round(1)

# Sort by ROI
current_spend = current_spend.sort_values('roi_score', ascending=False)

print("📊 Budget Allocation Analysis:\n")
print(current_spend[['source', 'roi_score', 'current_allocation_%', 'recommended_allocation_%', 'change_$']])
print(f"\n💵 Total Recruiting Spend: ${total_spend:,.0f}")

In [ ]:
# Visualize current vs. recommended allocation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Current allocation
ax1.barh(current_spend['source'], current_spend['current_allocation_%'], color='steelblue')
ax1.set_xlabel('% of Budget')
ax1.set_title('Current Budget Allocation', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Recommended allocation
colors = ['green' if x > 0 else 'red' for x in current_spend['change_$']]
ax2.barh(current_spend['source'], current_spend['recommended_allocation_%'], color=colors, alpha=0.7)
ax2.set_xlabel('% of Budget')
ax2.set_title('Recommended Budget Allocation (ROI-Optimized)', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Green = Increase investment | Red = Decrease investment")

## 7. Key Findings & Recommendations

In [ ]:
# Generate automated insights
print("="*70)
print("🎯 KEY FINDINGS")
print("="*70)

# Top quality source
top_quality = quality_by_source.iloc[0]
print(f"\n1️⃣ HIGHEST QUALITY SOURCE")
print(f"   {quality_by_source.index[0]}")
print(f"   • Performance: {top_quality['avg_performance']:.2f}/5")
print(f"   • Retention: {top_quality['retention_rate']:.1f}%")
print(f"   • Manager Satisfaction: {top_quality['avg_satisfaction']:.2f}/5")

# Best ROI source
best_roi = roi_analysis.iloc[0]
print(f"\n2️⃣ BEST ROI SOURCE")
print(f"   {roi_analysis.index[0]}")
print(f"   • Cost per Hire: ${best_roi['cost_per_hire']:,.0f}")
print(f"   • Quality Score: {best_roi['quality_score']:.1f}/100")
print(f"   • ROI Score: {best_roi['roi_score']:.2f}")

# Highest volume source
top_volume = source_metrics.iloc[0]
print(f"\n3️⃣ HIGHEST VOLUME SOURCE")
print(f"   {source_metrics.index[0]}")
print(f"   • Applications: {int(top_volume['applications']):,}")
print(f"   • Hires: {int(top_volume['hires'])}")
print(f"   • Conversion Rate: {top_volume['hire_rate_%']:.2f}%")

# Best conversion rate
best_conversion = source_metrics.sort_values('hire_rate_%', ascending=False).iloc[0]
print(f"\n4️⃣ BEST CONVERSION RATE")
print(f"   {source_metrics.sort_values('hire_rate_%', ascending=False).index[0]}")
print(f"   • Hire Rate: {best_conversion['hire_rate_%']:.2f}%")
print(f"   • vs. Overall: {(source_metrics['hires'].sum() / source_metrics['applications'].sum() * 100):.2f}%")

print("\n" + "="*70)
print("💡 RECOMMENDATIONS")
print("="*70)

# Identify sources to increase/decrease
increase_sources = current_spend[current_spend['change_$'] > 1000].sort_values('change_$', ascending=False)
decrease_sources = current_spend[current_spend['change_$'] < -1000].sort_values('change_$')

if len(increase_sources) > 0:
    print(f"\n✅ INCREASE INVESTMENT:")
    for idx, row in increase_sources.head(3).iterrows():
        print(f"   • {row['source']}: +${row['change_$']:,.0f} (+{row['change_%']:.0f}%)")
        print(f"     → ROI Score: {row['roi_score']:.2f} | Quality: {row['quality_score']:.1f}/100")

if len(decrease_sources) > 0:
    print(f"\n⚠️ DECREASE INVESTMENT:")
    for idx, row in decrease_sources.head(3).iterrows():
        print(f"   • {row['source']}: {row['change_$']:,.0f} ({row['change_%']:.0f}%)")
        print(f"     → ROI Score: {row['roi_score']:.2f} | Quality: {row['quality_score']:.1f}/100")

print(f"\n🎯 STRATEGIC ACTIONS:")
print(f"   1. Prioritize high-ROI sources with room to grow")
print(f"   2. Maintain source diversification (no single source >40%)")
print(f"   3. Investigate why top quality sources work well")
print(f"   4. Improve processes for high-volume, low-quality sources")
print(f"   5. Track quality metrics quarterly to adjust allocation")

print("\n" + "="*70)

## Next Steps

1. **Validate Findings**: Review with recruiting team and hiring managers
2. **Implement Changes**: Gradually shift budget allocation over 2-3 quarters
3. **Monitor Impact**: Track quality metrics monthly to measure improvement
4. **Investigate Drivers**: Conduct qualitative research on why top sources perform well
5. **Iterate**: Revisit analysis quarterly and adjust strategy

## Related Analyses
- `02_time_to_fill_analysis.ipynb` - Process bottlenecks
- `03_quality_of_hire_prediction.ipynb` - Predictive modeling
- `05_diversity_pipeline.ipynb` - Source diversity analysis